In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd

from mixedimputer import TimeSeriesImputer
from mixedimputer.corrupt_data import DataCorrupter

warnings.filterwarnings("ignore")


In [ ]:
# ── 1. Create synthetic vehicle telemetry dataset ──

np.random.seed(42)
n = 100

# Use numeric time (seconds) — MixedImputer requires all columns to be
# either numeric or categorical; datetime64 is neither.
time = np.arange(n, dtype=float)  # 0, 1, 2, ..., 99 seconds

# Speed: sinusoidal cruise with noise (auto-correlated via cumsum of small steps)
speed_base = 50 + 15 * np.sin(np.linspace(0, 3 * np.pi, n))  # oscillate 35–65 km/h
speed = speed_base + np.random.normal(0, 2, n).cumsum() * 0.05  # slight drift
speed = np.clip(speed, 0, 130)

# Engine temperature: gradually warming up, periodic cooling, + noise
temp_base = 80 + 10 * np.sin(np.linspace(0, 2 * np.pi, n))  # oscillate 70–90 °C
temp = temp_base + np.random.normal(0, 1, n).cumsum() * 0.02  # slight drift
temp = np.clip(temp, 60, 110)

df_original = pd.DataFrame({
    "timestamp": time,
    "speed_kmh": speed,
    "engine_temp_c": temp,
})

df_original.head(10)


In [ ]:
# ── 2. Corrupt ~15% per numeric column (MCAR) ──

corrupter = DataCorrupter(
    mechanism="MCAR",
    corruption_fraction=0.15,
    numeric_columns=["speed_kmh", "engine_temp_c"],
    random_state=42,
)

df_corrupted, mask, df_original_copy = corrupter.corrupt(df_original)

# Show corruption stats
total_cells = mask[["speed_kmh", "engine_temp_c"]].size
corrupted_cells = mask[["speed_kmh", "engine_temp_c"]].sum().sum()
print(f"Total cells (speed + temp): {total_cells}")
print(f"Corrupted cells:              {corrupted_cells}")
print(f"Actual corruption rate:        {corrupted_cells / total_cells:.2%}")
print()
print("First 10 rows — corrupted (NaN where values were removed):")
df_corrupted.head(10)


In [ ]:
# ── 3. Impute with TimeSeriesImputer ──

imputer = TimeSeriesImputer(
    time_column="timestamp",
    lags=[1, 2, 3],
    rolling_windows=[5, 10],
    initial_strategy="forward_fill",
    max_iter=10,
    random_state=42,
    verbose=0,
)

df_imputed = imputer.fit_transform(df_corrupted)

print("Imputation complete.")
df_imputed.head(10)


In [ ]:
# ── 4. Direct comparison: original vs imputed ──

comparisons = []

for col in ["speed_kmh", "engine_temp_c"]:
    corrupted_idx = mask.index[mask[col]]  # rows where this column was corrupted
    for idx in corrupted_idx:
        comparisons.append({
            "column": col,
            "timestamp": df_original.loc[idx, "timestamp"],
            "original": df_original.loc[idx, col],
            "imputed": df_imputed.loc[idx, col],
            "absolute_error": abs(df_original.loc[idx, col] - df_imputed.loc[idx, col]),
        })

df_comp = pd.DataFrame(comparisons)

# --- Summary statistics ---
print("=" * 70)
print("DIRECT COMPARISON — ORIGINAL vs. IMPUTED (corrupted positions only)")
print("=" * 70)

for col in ["speed_kmh", "engine_temp_c"]:
    sub = df_comp[df_comp["column"] == col]
    mae = sub["absolute_error"].mean()
    rmse = (sub["absolute_error"] ** 2).mean() ** 0.5
    n_corrupted = len(sub)
    print(f"\n--- {col} ({n_corrupted} corrupted values) ---")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")

# --- Show every corrupted position ---
print("\n" + "=" * 70)
print("ROW-BY-ROW COMPARISON (all corrupted positions)")
print("=" * 70)
display(df_comp)


In [ ]:
# ── 5. Interactive visual (Plotly) ──

import plotly.graph_objects as go
from plotly.subplots import make_subplots

TEMPLATE = "plotly_white"

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Speed (km/h)", "Engine Temperature (°C)"),
    vertical_spacing=0.08,
)

for row, col in enumerate(["speed_kmh", "engine_temp_c"], start=1):
    corrupt_idx = mask.index[mask[col]]

    # Full original line
    fig.add_trace(
        go.Scatter(
            x=df_original["timestamp"], y=df_original[col],
            mode="lines", name=f"{col} — original",
            line=dict(color="black", width=1), opacity=0.5,
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Corrupted original values (red markers)
    fig.add_trace(
        go.Scatter(
            x=df_original.loc[corrupt_idx, "timestamp"],
            y=df_original.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — original (corrupted)",
            marker=dict(color="red", size=8, symbol="circle"),
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Imputed values (blue X markers)
    fig.add_trace(
        go.Scatter(
            x=df_imputed.loc[corrupt_idx, "timestamp"],
            y=df_imputed.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — imputed",
            marker=dict(color="blue", size=8, symbol="x"),
            legendgroup=col,
        ),
        row=row, col=1,
    )

fig.update_xaxes(title_text="Timestamp (seconds)", row=2, col=1)
fig.update_layout(
    title="TimeSeriesImputer: Original vs. Imputed (red = removed, blue x = imputed)",
    template=TEMPLATE,
    height=600,
    hovermode="x unified",
)
fig.show()


# TimeSeriesImputer — Direct Comparison Benchmark

Create a synthetic 100-second vehicle telemetry dataset, corrupt 5% of values per column, impute with `TimeSeriesImputer`, and compare imputed values against the original ground truth.

# ── GPU Sensor Log Benchmark ──

Load real-world GPU sensor data logged during a gaming session, corrupt two key columns, impute with `TimeSeriesImputer`, and compare against ground truth.

In [ ]:
# ── 6. Load & prepare GPU sensor log ──

# CSV with padded whitespace; column names contain non-UTF-8 characters.
df_raw = pd.read_csv(
    "../data/gpu_sensor_log.txt",
    skipinitialspace=True,
    encoding="latin1",
)

# Strip whitespace from column names (they have leading/trailing spaces)
df_raw.columns = df_raw.columns.str.strip()

# Parse the Date column to datetime
df_raw["Date"] = pd.to_datetime(df_raw["Date"])

# Convert datetime to numeric seconds-from-start for TimeSeriesImputer
df_raw["time_sec"] = (df_raw["Date"] - df_raw["Date"].iloc[0]).dt.total_seconds().astype(float)

print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"Time range: {df_raw['Date'].iloc[0]} → {df_raw['Date'].iloc[-1]}")
print(f"Duration: {df_raw['time_sec'].iloc[-1]:.0f} seconds")
print(f"\nColumns: {list(df_raw.columns)}")
df_raw.head()


In [ ]:
# ── 7. Select columns & corrupt ──

TARGET_COLS = ["GPU Temperature [°C]", "Fan Speed (RPM) [RPM]"]

# Build a clean DataFrame with just the columns we need
df_gpu = df_raw[["time_sec"] + TARGET_COLS].copy()

# Corrupt ~10% of each target column
corrupter_gpu = DataCorrupter(
    mechanism="MCAR",
    corruption_fraction=0.1,
    numeric_columns=TARGET_COLS,
    random_state=42,
)

df_gpu_corrupted, mask_gpu, df_gpu_original = corrupter_gpu.corrupt(df_gpu)

# Show corruption stats
for col in TARGET_COLS:
    n_corrupt = mask_gpu[col].sum()
    n_total = len(df_gpu)
    print(f"{col}: {n_corrupt}/{n_total} corrupted ({n_corrupt/n_total:.1%})")

print("\nFirst 10 rows — corrupted:")
df_gpu_corrupted.head(10)


In [ ]:
# ── 8. Impute GPU data ──

imputer_gpu = TimeSeriesImputer(
    time_column="time_sec",
    lags=[1, 2, 3, 5],
    rolling_windows=[10, 30],
    initial_strategy="forward_fill",
    max_iter=10,
    random_state=42,
    verbose=0,
)

df_gpu_imputed = imputer_gpu.fit_transform(df_gpu_corrupted)

print("Imputation complete.")
df_gpu_imputed.head(10)


In [ ]:
# ── 9. Direct comparison: original vs imputed ──

comparisons_gpu = []

for col in TARGET_COLS:
    corrupted_idx = mask_gpu.index[mask_gpu[col]]
    for idx in corrupted_idx:
        comparisons_gpu.append({
            "column": col,
            "time_sec": df_gpu_original.loc[idx, "time_sec"],
            "original": df_gpu_original.loc[idx, col],
            "imputed": df_gpu_imputed.loc[idx, col],
            "absolute_error": abs(df_gpu_original.loc[idx, col] - df_gpu_imputed.loc[idx, col]),
        })

df_comp_gpu = pd.DataFrame(comparisons_gpu)

# --- Summary statistics ---
print("=" * 70)
print("GPU SENSOR LOG — DIRECT COMPARISON (corrupted positions only)")
print("=" * 70)

for col in TARGET_COLS:
    sub = df_comp_gpu[df_comp_gpu["column"] == col]
    mae = sub["absolute_error"].mean()
    rmse = (sub["absolute_error"] ** 2).mean() ** 0.5
    max_err = sub["absolute_error"].max()
    n_corrupted = len(sub)
    # Also show the scale of the data for context
    col_min = df_gpu_original[col].min()
    col_max = df_gpu_original[col].max()
    col_range = col_max - col_min
    print(f"\n--- {col} ({n_corrupted} corrupted values) ---")
    print(f"  Data range: {col_min:.1f} – {col_max:.1f} (span: {col_range:.1f})")
    print(f"  MAE:  {mae:.4f}  ({mae/col_range*100:.2f}% of range)")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  Max error: {max_err:.4f}")

# --- Show every corrupted position ---
print("\n" + "=" * 70)
print("ROW-BY-ROW COMPARISON (all corrupted positions)")
print("=" * 70)
display(df_comp_gpu)


In [ ]:
# ── 10. Interactive visual (GPU data) ──

fig_gpu = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("GPU Temperature (°C)", "Fan Speed (RPM)"),
    vertical_spacing=0.08,
)

for row, col in enumerate(TARGET_COLS, start=1):
    corrupt_idx = mask_gpu.index[mask_gpu[col]]

    # Full original line
    fig_gpu.add_trace(
        go.Scatter(
            x=df_gpu_original["time_sec"], y=df_gpu_original[col],
            mode="lines", name=f"{col} — original",
            line=dict(color="black", width=1), opacity=0.5,
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Corrupted original values (red markers)
    fig_gpu.add_trace(
        go.Scatter(
            x=df_gpu_original.loc[corrupt_idx, "time_sec"],
            y=df_gpu_original.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — original (corrupted)",
            marker=dict(color="red", size=6, symbol="circle"),
            legendgroup=col,
        ),
        row=row, col=1,
    )

    # Imputed values (blue X markers)
    fig_gpu.add_trace(
        go.Scatter(
            x=df_gpu_imputed.loc[corrupt_idx, "time_sec"],
            y=df_gpu_imputed.loc[corrupt_idx, col],
            mode="markers", name=f"{col} — imputed",
            marker=dict(color="blue", size=8, symbol="x",
                        line=dict(width=1.5)),
            legendgroup=col,
        ),
        row=row, col=1,
    )

fig_gpu.update_xaxes(title_text="Time (seconds since start)", row=2, col=1)
fig_gpu.update_layout(
    title="GPU Sensor Log — TimeSeriesImputer: Original vs. Imputed (red = removed, blue x = imputed)",
    template=TEMPLATE,
    height=600,
    hovermode="x unified",
)
fig_gpu.show()


# ── GPU Sensor Log — Multi-Level Benchmark ──

Evaluate `TimeSeriesImputer` against forward-fill and mean-imputation baselines
at **1%**, **10%**, and **30%** corruption on the two GPU sensor columns.

In [ ]:
# ── 11. Benchmark function ──

from sklearn.metrics import mean_squared_error

def run_ts_benchmark(df_clean: pd.DataFrame, fraction: float,
                     target_cols: list, time_col: str = "time_sec",
                     random_state: int = 42) -> dict:
    """Corrupt, impute, and evaluate for a single corruption fraction."""

    # 1. Corrupt the target columns
    corrupter = DataCorrupter(
        mechanism="MCAR",
        corruption_fraction=fraction,
        numeric_columns=target_cols,
        random_state=random_state,
    )
    corrupted, mask, original = corrupter.corrupt(df_clean)

    # 2. Impute with TimeSeriesImputer
    imputer = TimeSeriesImputer(
        time_column=time_col,
        lags=[1, 2, 3, 5],
        rolling_windows=[10, 30],
        initial_strategy="forward_fill",
        max_iter=10,
        random_state=random_state,
        verbose=0,
    )
    imputed = imputer.fit_transform(corrupted)

    # 3. Baselines: forward-fill and mean imputation
    corrupted_ffill = corrupted.copy()
    corrupted_mean = corrupted.copy()
    for col in target_cols:
        corrupted_ffill[col] = corrupted_ffill[col].ffill().bfill()
        corrupted_mean[col] = corrupted_mean[col].fillna(original[col].mean())

    # 4. Evaluate per column
    results = {"fraction": fraction}

    for col in target_cols:
        m = mask[col]
        if m.sum() == 0:
            results[f"{col}_rmse"] = np.nan
            results[f"{col}_mae"] = np.nan
            continue

        true_vals = original.loc[m, col].to_numpy(dtype=float)
        imp_vals  = imputed.loc[m, col].to_numpy(dtype=float)
        ff_vals   = corrupted_ffill.loc[m, col].to_numpy(dtype=float)
        mean_vals = corrupted_mean.loc[m, col].to_numpy(dtype=float)

        rmse_imp  = np.sqrt(mean_squared_error(true_vals, imp_vals))
        rmse_ff   = np.sqrt(mean_squared_error(true_vals, ff_vals))
        rmse_mean = np.sqrt(mean_squared_error(true_vals, mean_vals))
        mae_imp   = np.mean(np.abs(true_vals - imp_vals))
        mae_ff    = np.mean(np.abs(true_vals - ff_vals))
        mae_mean  = np.mean(np.abs(true_vals - mean_vals))

        results[f"{col}_rmse"]         = round(rmse_imp, 2)
        results[f"{col}_rmse_ffill"]   = round(rmse_ff, 2)
        results[f"{col}_rmse_mean"]    = round(rmse_mean, 2)
        results[f"{col}_mae"]          = round(mae_imp, 2)
        results[f"{col}_mae_ffill"]    = round(mae_ff, 2)
        results[f"{col}_mae_mean"]     = round(mae_mean, 2)

    results["n_corrupted_total"] = int(mask[target_cols].sum().sum())
    return results

print("TS benchmark function defined")


In [ ]:
# ── 12. Run all 3 scenarios ──

FRACTIONS = [0.01, 0.10, 0.30]
SEED = 42

all_ts_results = []
for frac in FRACTIONS:
    print(f"Running {frac:.0%} corruption...", end=" ")
    row = run_ts_benchmark(
        df_gpu,              # the clean GPU DataFrame (time_sec + TARGET_COLS)
        fraction=frac,
        target_cols=TARGET_COLS,
        random_state=SEED,
    )
    all_ts_results.append(row)
    n_c = row["n_corrupted_total"]
    print(f"({n_c} cells corrupted)")

# Build results DataFrame
results_ts_df = pd.DataFrame(all_ts_results).set_index("fraction")
results_ts_df.index = results_ts_df.index.map(lambda x: f"{x:.0%}")
results_ts_df.index.name = "Corruption Level"

print("\nAll TS scenarios complete\n")
results_ts_df


## 13. Visualizations

### 13a. RMSE — TimeSeriesImputer vs. Baselines

Lower is better. Dashed lines show forward-fill and mean-imputation baselines.

In [ ]:
# ── RMSE bar chart: TimeSeriesImputer vs. baselines ──
import plotly.graph_objects as go

rmse_imp_cols = [c for c in results_ts_df.columns if c.endswith("_rmse") and "_ffill" not in c and "_mean" not in c]
rmse_imp_data = results_ts_df[rmse_imp_cols].copy()
rmse_imp_data.columns = [c.replace("_rmse", "") for c in rmse_imp_cols]

COLORS_TS = ["#66c2a5", "#fc8d62", "#8da0cb"]  # green, orange, purple

fig_ts_rmse = go.Figure()

# TimeSeriesImputer bars
for i, col in enumerate(rmse_imp_data.columns):
    fig_ts_rmse.add_trace(go.Bar(
        name=f"{col} (TimeSeriesImputer)",
        x=rmse_imp_data.index,
        y=rmse_imp_data[col],
        marker_color=COLORS_TS[0],
        text=rmse_imp_data[col].round(1),
        textposition="outside",
    ))

# Forward-fill baseline (dashed scatter)
ff_cols = [c for c in results_ts_df.columns if c.endswith("_rmse_ffill")]
ff_data = results_ts_df[ff_cols].copy()
ff_data.columns = [c.replace("_rmse_ffill", "") for c in ff_cols]

for i, col in enumerate(ff_data.columns):
    fig_ts_rmse.add_trace(go.Scatter(
        name=f"{col} (Forward-fill baseline)",
        x=ff_data.index,
        y=ff_data[col],
        mode="lines+markers+text",
        marker=dict(color=COLORS_TS[1], size=10, symbol="diamond"),
        line=dict(dash="dash", color=COLORS_TS[1]),
        text=ff_data[col].round(1),
        textposition="top center",
    ))

# Mean-imputation baseline (dotted scatter)
mean_cols = [c for c in results_ts_df.columns if c.endswith("_rmse_mean")]
mean_data = results_ts_df[mean_cols].copy()
mean_data.columns = [c.replace("_rmse_mean", "") for c in mean_cols]

for i, col in enumerate(mean_data.columns):
    fig_ts_rmse.add_trace(go.Scatter(
        name=f"{col} (Mean baseline)",
        x=mean_data.index,
        y=mean_data[col],
        mode="lines+markers+text",
        marker=dict(color=COLORS_TS[2], size=10, symbol="cross"),
        line=dict(dash="dot", color=COLORS_TS[2]),
        text=mean_data[col].round(1),
        textposition="bottom center",
    ))

fig_ts_rmse.update_layout(
    title="GPU Sensor Log — RMSE: TimeSeriesImputer vs. Baselines",
    xaxis_title="Corruption Level",
    yaxis_title="RMSE (lower is better)",
    barmode="group",
    template="plotly_white",
    height=500,
)
fig_ts_rmse.show()
